# Hybrid RAG with a Steerable Jev Reranker — A Real PDF Pipeline

This notebook builds an end-to-end **Retrieval-Augmented Generation** pipeline over real PDF
documents (including PDFs containing embedded images/diagrams), and then measures what a
**Jev** reranker (TypeSafe AI's `typesafe-sdk`, using the `Noul` calibrated-probability
primitive) actually buys you on top of plain hybrid retrieval.

**Stack**
| Component | Choice |
|---|---|
| Document processing | PyMuPDF (text) + Claude Sonnet vision captioning (embedded images) |
| Chunking | LangChain `RecursiveCharacterTextSplitter` |
| Vector store | Chroma (`langchain-chroma`), local persistence |
| Sparse retriever | BM25 (`rank_bm25` via `langchain_community.retrievers.BM25Retriever`) |
| Hybrid retrieval | LangChain `EnsembleRetriever` (BM25 + vector, reciprocal-rank fusion) |
| Reranker | **Jev** (`typesafe-sdk`, `Noul` primitive) — steerable, calibrated, cheap |
| LLM (synthesis + judge + vision) | Claude Sonnet, called **through OpenRouter**'s OpenAI-compatible API |
| Embeddings | Local `BAAI/bge-small-en-v1.5` (`sentence-transformers`, no extra API key needed) |

**Corpus.** Two real PDFs sit next to this notebook and are used as the knowledge base:
- `Understanding_JEV_plain.pdf` — a plain, text-only PDF (4 pages, no images).
- `To Data & Beyond - Jev Clearly Explained_ I Built a Decision Layer for an AI Agent.pdf` —
  a 12-page PDF with 6 embedded diagrams, exercising the image-handling path.

Both explain the same subject (the Jev / TypeSafe decision primitives), which is convenient:
the reranker built in this notebook is evaluated on documents *about the exact technique it
implements*.

**What gets measured at the end.** For a held-out set of questions grounded in these documents,
every query is run through two pipelines — **Hybrid RAG** and **Hybrid RAG + Jev rerank** — and
we compare:
1. **Latency** (retrieval / rerank / generation, broken out and totalled)
2. **Response faithfulness** (an LLM-judge score: is the answer fully supported by its context?)
3. **Token usage** (context + completion tokens sent to the synthesis LLM)

Every concept below lives in its own cell so each piece can be read, run, and modified in isolation.


---
## 1. Environment Setup

Dependencies are installed once, idempotently — if a package is already importable, the install
is skipped. Everything here is a real, third-party PyPI package (`typesafe-sdk` is TypeSafe AI's
official Jev SDK).

In [1]:
import importlib, subprocess, sys

def ensure(module: str, pip_name: str | None = None) -> None:
    try:
        importlib.import_module(module)
    except ImportError:
        print(f"installing {pip_name or module} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True)
        importlib.invalidate_caches()

for module, pip_name in [
    ("langchain_community", "langchain-community"),
    ("langchain_text_splitters", "langchain-text-splitters"),
    ("langchain_chroma", "langchain-chroma"),
    ("langchain_huggingface", "langchain-huggingface"),
    ("langchain_openai", "langchain-openai"),
    ("langchain_classic", "langchain-classic"),
    ("langchain_core", "langchain-core"),
    ("chromadb", "chromadb"),
    ("rank_bm25", "rank-bm25"),
    ("fitz", "pymupdf"),
    ("sentence_transformers", "sentence-transformers"),
    ("typesafe_sdk", "typesafe-sdk"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("numpy", "numpy"),
]:
    ensure(module, pip_name)

print("All dependencies present.")


All dependencies present.


In [2]:
import os, re, json, time, base64, textwrap, warnings, getpass
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
print("Core imports ready.")


Core imports ready.


### Chart Styling

One small styling layer, applied once, so every performance chart in this notebook (Sections 10b
and 16) shares the same palette, value-labeled bars, and optional threshold lines -- the same
visual language used for the benchmark charts in `JEV_RERANKER_RAG.ipynb`.

In [3]:
PALETTE = {
    "ink": "#0F172A", "ink2": "#334155", "muted": "#64748B",
    "grid": "#F1F5F9", "border": "#E2E8F0", "surface": "#FFFFFF", "plane": "#F8FAFC",
    "no_jev": "#94A3B8", "with_jev": "#2563EB",
    "good": "#059669", "warning": "#D97706", "critical": "#DC2626",
}
SERIES = ["#2563EB", "#D97706", "#059669", "#7C3AED", "#DB2777"]

plt.rcParams.update({
    "figure.facecolor": PALETTE["plane"], "savefig.facecolor": PALETTE["plane"], "figure.dpi": 120,
    "axes.facecolor": PALETTE["surface"], "axes.edgecolor": PALETTE["border"], "axes.linewidth": 1.0,
    "axes.labelcolor": PALETTE["ink2"], "axes.titlecolor": PALETTE["ink"], "axes.labelsize": 10,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlesize": 12.5, "axes.titlepad": 14,
    "axes.grid": True, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": PALETTE["grid"], "grid.linewidth": 0.9,
    "xtick.color": PALETTE["muted"], "ytick.color": PALETTE["muted"],
    "text.color": PALETTE["ink2"], "legend.frameon": False, "legend.labelcolor": PALETTE["ink2"],
    "font.family": "sans-serif",
})

def finish(ax, title=None, sub=None):
    """Applies a title + optional subtitle in the notebook's consistent chart style."""
    if title:
        ax.set_title(title, pad=20 if sub else 10)
    if sub:
        ax.text(0, 1.02, sub, transform=ax.transAxes, color=PALETTE["muted"], fontsize=9, va="bottom", ha="left")

def draw_labeled_bars(ax, labels, values, colors=None, threshold=None, threshold_label=None,
                       xlabel="", fmt="{:.2f}", xlim=None):
    """Horizontal bars with inline value labels and an optional threshold line -- the recurring
    chart shape used throughout this notebook's performance measurements."""
    y = np.arange(len(labels))
    if colors is None:
        colors = [SERIES[0]] * len(labels)
    ax.barh(y, values, color=colors, height=0.55, zorder=2)
    span = (max(values) - min(values)) if len(values) > 1 else max(values, default=1)
    for i, v in enumerate(values):
        ax.text(v + span * 0.015, i, f" {fmt.format(v)}", va="center", ha="left",
                fontsize=9.5, weight="bold", color=PALETTE["ink"], zorder=3)
    if threshold is not None:
        ax.axvline(threshold, color=PALETTE["muted"], lw=1.2, ls="--", zorder=1)
        if threshold_label:
            ax.text(threshold, len(labels) - 0.4, f" {threshold_label}", color=PALETTE["muted"],
                    fontsize=8.5, va="bottom")
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    if xlim:
        ax.set_xlim(xlim)
    if xlabel:
        ax.set_xlabel(xlabel)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)

print("Chart styling helpers ready.")


Chart styling helpers ready.


---
## 2. Configuration & API Clients

Two external services are used:
- **OpenRouter** — an OpenAI-compatible gateway used to reach **Claude Sonnet** for response
  synthesis, image captioning, and faithfulness judging. Because OpenRouter speaks the OpenAI
  wire format, LangChain's `ChatOpenAI` class is pointed at OpenRouter's `base_url` instead of
  OpenAI's — no separate OpenRouter-specific client is needed.
- **TypeSafe AI (Jev)** — reached via the official `typesafe-sdk`, used for the reranking step.

Keys are read from the environment first and only prompted for interactively if missing, so this
notebook is safe to re-run without re-entering secrets.

In [4]:
def get_key(env_var: str) -> str:
    val = os.environ.get(env_var)
    if val:
        return val
    val = getpass.getpass(f"{env_var}: ")
    os.environ[env_var] = val
    return val

OPENROUTER_API_KEY = get_key("OPENROUTER_API_KEY")
TYPESAFE_API_KEY = get_key("TYPESAFE_API_KEY")

# OpenRouter model slug for Claude Sonnet. Verify the exact current slug for your account at
# https://openrouter.ai/models -- OpenRouter's naming can trail Anthropic's own model names.
OPENROUTER_MODEL = "anthropic/claude-sonnet-4.5"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

print("Keys loaded. Using OpenRouter model:", OPENROUTER_MODEL)


Keys loaded. Using OpenRouter model: anthropic/claude-sonnet-4.5


In [5]:
from langchain_openai import ChatOpenAI

# Synthesis LLM: writes the final RAG answer.
llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    temperature=0.2,
    max_tokens=800,
    default_headers={
        "HTTP-Referer": "https://github.com/jev-rag-poc",  # OpenRouter analytics field, optional
        "X-Title": "Hybrid RAG with Jev Reranker POC",
    },
)

# Vision LLM: captions embedded PDF images. Same model/route -- Claude Sonnet is multimodal.
vision_llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    temperature=0.0,
    max_tokens=300,
)

# Judge LLM: scores faithfulness. Kept separate (temperature 0) for stable grading.
judge_llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    temperature=0.0,
    max_tokens=300,
)

print("OpenRouter-backed Claude Sonnet clients ready (synthesis, vision, judge).")


OpenRouter-backed Claude Sonnet clients ready (synthesis, vision, judge).


In [6]:
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score

jev_client = TypeSafeClient(api_key=TYPESAFE_API_KEY, timeout=60.0)

# --- lightweight usage metering, mirrors what the Jev pricing model bills on ---
JEV_USAGE = {"calls": 0, "input_tokens": 0, "latency_s": 0.0}
JEV_RATE_PER_1M_INPUT = 0.042  # USD, per Jev's published pricing for Noul evaluations

def jev_ask(state, questions):
    t0 = time.perf_counter()
    r = jev_client.system_one(state=state, questions=questions)
    JEV_USAGE["calls"] += 1
    JEV_USAGE["latency_s"] += time.perf_counter() - t0
    JEV_USAGE["input_tokens"] += r.usage.input_tokens
    return r

def jev_fanout(states, questions, workers: int = 16):
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return list(ex.map(lambda s: jev_ask(s, questions), states))

print("Jev (typesafe-sdk) client ready.")


Jev (typesafe-sdk) client ready.


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

# Local, free embedding model -- keeps the pipeline self-contained (no embeddings API key needed).
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

print(f"Embedding model '{EMBEDDING_MODEL_NAME}' loaded (dim={len(embeddings.embed_query('probe'))}).")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model 'BAAI/bge-small-en-v1.5' loaded (dim=384).


---
## 3. Document Processing — Plain PDF

`PyMuPDFLoader` extracts one LangChain `Document` per page, carrying page-level metadata
(`source`, `page`, `total_pages`, ...) for free. This is the straightforward case: pure text,
no embedded images to handle.

In [8]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

PLAIN_PDF_PATH = "Understanding_JEV_plain.pdf"

plain_pages = PyMuPDFLoader(PLAIN_PDF_PATH).load()
for d in plain_pages:
    d.metadata["content_type"] = "text"

print(f"Loaded {len(plain_pages)} pages from '{PLAIN_PDF_PATH}'")
print("---")
print(plain_pages[0].page_content[:300])


Loaded 4 pages from 'Understanding_JEV_plain.pdf'
---
Understanding JEV: Core Decision-Making Primitives in RAG Pipelines
1. Introduction: The Need for Decision Points in RAG
Modern Retrieval-Augmented Generation (RAG) pipelines are designed to retrieve relevant
information from a knowledge base and feed it to a Language Model (LLM) to generate accurat


---
## 4. Document Processing — PDF with Embedded Images

Real PDFs are rarely pure text. `To Data & Beyond - Jev Clearly Explained...pdf` embeds 6
diagrams across its 12 pages. Handling this properly means two parallel extraction paths on the
*same* file:

1. **Text path** — `PyMuPDFLoader`, exactly as above, one `Document` per page.
2. **Image path** — `PyMuPDF` (`fitz`) pulls each embedded raster image out of the PDF; each
   image is sent to **Claude Sonnet's vision** (through OpenRouter) to produce a short caption,
   which becomes its own `Document` with `content_type="image_caption"` metadata. This makes a
   diagram's content retrievable by a text query even though the diagram itself has no text layer.

Tiny embedded images (icons, logos, decorative rules) are filtered out by a minimum pixel-dimension
threshold so they don't waste vision calls or pollute the index with noise.

In [9]:
IMAGE_PDF_PATH = "To Data & Beyond - Jev Clearly Explained_ I Built a Decision Layer for an AI Agent.pdf"

image_pages = PyMuPDFLoader(IMAGE_PDF_PATH).load()
for d in image_pages:
    d.metadata["content_type"] = "text"

print(f"Loaded {len(image_pages)} text pages from '{IMAGE_PDF_PATH}'")


Loaded 12 text pages from 'To Data & Beyond - Jev Clearly Explained_ I Built a Decision Layer for an AI Agent.pdf'


In [10]:
import fitz  # PyMuPDF

def extract_embedded_images(pdf_path: str, min_dim: int = 80) -> list[dict]:
    """Pulls every embedded raster image out of a PDF, skipping small icons/logos."""
    doc = fitz.open(pdf_path)
    extracted = []
    for page_no in range(doc.page_count):
        for img in doc[page_no].get_images(full=True):
            xref = img[0]
            base_image = doc.extract_image(xref)
            width, height = base_image.get("width", 0), base_image.get("height", 0)
            if width < min_dim or height < min_dim:
                continue
            extracted.append({
                "page": page_no,
                "ext": base_image["ext"],
                "bytes": base_image["image"],
                "width": width,
                "height": height,
            })
    doc.close()
    return extracted

raw_images = extract_embedded_images(IMAGE_PDF_PATH)
print(f"Extracted {len(raw_images)} content-sized images from '{IMAGE_PDF_PATH}':")
for im in raw_images:
    print(f"  page {im['page']:>2} | {im['width']}x{im['height']} | .{im['ext']}")


Extracted 5 content-sized images from 'To Data & Beyond - Jev Clearly Explained_ I Built a Decision Layer for an AI Agent.pdf':
  page  1 | 720x405 | .jpeg
  page  2 | 630x201 | .jpeg
  page  5 | 720x288 | .jpeg
  page  6 | 550x408 | .png
  page 10 | 720x288 | .jpeg


In [11]:
def caption_image(image_bytes: bytes, ext: str, page_no: int) -> str:
    """Sends one embedded image to Claude Sonnet (vision, via OpenRouter) for a short caption."""
    media_type = "image/jpeg" if ext.lower() in ("jpg", "jpeg") else f"image/{ext.lower()}"
    b64 = base64.b64encode(image_bytes).decode("utf-8")
    message = {
        "role": "user",
        "content": [
            {"type": "text", "text": (
                f"This image is a figure from page {page_no + 1} of a technical article about "
                "RAG reranking. In 2-3 sentences, describe what the figure conveys: the "
                "architecture, comparison, or data it shows. Be concrete, not generic."
            )},
            {"type": "image_url", "image_url": {"url": f"data:{media_type};base64,{b64}"}},
        ],
    }
    response = vision_llm.invoke([message])
    return response.content.strip()

image_captions = []
for im in raw_images:
    caption = caption_image(im["bytes"], im["ext"], im["page"])
    image_captions.append(
        Document(
            page_content=caption,
            metadata={
                "source": IMAGE_PDF_PATH,
                "page": im["page"],
                "content_type": "image_caption",
                "image_size": f"{im['width']}x{im['height']}",
            },
        )
    )
    print(f"[page {im['page']}] {caption}\n")

print(f"Captioned {len(image_captions)} images.")


[page 1] This figure illustrates a decision layer architecture for an AI agent using Jev, showing how agent state (including Goal, Context, Tools, and History) flows into Jev's evaluation system. Jev outputs three typed decisions with associated probabilities: "risky?" (p=0.12), "needs approval?" (p=0.28), and "relevant?" (p=0.91), which then route to three different actions: Allow, Review, or Block. The diagram demonstrates how probabilistic outputs and latency considerations enable agent-style workflow control with failure modes built in.

[page 2] This figure illustrates a RAG reranking architecture with three main stages: an initial retrieval step that fetches candidate documents, a reranking module that scores and reorders these candidates based on relevance to the query, and a final generation step that uses the top reranked documents to produce the output. The diagram shows how documents flow from left to right through retrieval, reranking (typically using a cross-encoder or sim

---
## 5. Combine the Corpus

Text pages from both PDFs and the generated image captions are merged into one flat list of
`Document`s -- the raw corpus that chunking will operate on.

In [12]:
raw_corpus = plain_pages + image_pages + image_captions

by_type = pd.Series([d.metadata["content_type"] for d in raw_corpus]).value_counts()
by_source = pd.Series([Path(d.metadata["source"]).name for d in raw_corpus]).value_counts()

print(f"Raw corpus: {len(raw_corpus)} documents")
print("\nBy content_type:\n", by_type.to_string())
print("\nBy source:\n", by_source.to_string())


Raw corpus: 21 documents

By content_type:
 text             16
image_caption     5

By source:
 To Data & Beyond - Jev Clearly Explained_ I Built a Decision Layer for an AI Agent.pdf    17
Understanding_JEV_plain.pdf                                                                4


---
## 6. Chunking

Page-level `Document`s are too coarse for retrieval (a whole page rarely matches a query's
specific granularity) and too fine for image captions (already atomic). `RecursiveCharacterTextSplitter`
splits **text** documents into overlapping chunks; **image captions are left intact** since
splitting a 2-3 sentence caption would only fragment it.

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 700
CHUNK_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

text_docs = [d for d in raw_corpus if d.metadata["content_type"] == "text"]
caption_docs = [d for d in raw_corpus if d.metadata["content_type"] == "image_caption"]

text_chunks = splitter.split_documents(text_docs)
all_chunks = text_chunks + caption_docs  # captions pass through unsplit

print(f"{len(text_docs)} text pages -> {len(text_chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
print(f"{len(caption_docs)} image captions kept as {len(caption_docs)} atomic chunks")
print(f"Total indexable chunks: {len(all_chunks)}")


16 text pages -> 47 chunks (size=700, overlap=100)
5 image captions kept as 5 atomic chunks
Total indexable chunks: 52


---
## 7. Metadata Enrichment

Every chunk gets a stable `chunk_id`, its position within the document, and a character count.
This metadata is what later lets the comparison stage report *which* chunks a pipeline actually
used, and lets a reader trace any answer back to a specific page and PDF.

In [14]:
for i, chunk in enumerate(all_chunks):
    chunk.metadata["chunk_id"] = f"chunk_{i:04d}"
    chunk.metadata["char_len"] = len(chunk.page_content)
    chunk.metadata.setdefault("page", -1)
    chunk.metadata["source_name"] = Path(chunk.metadata["source"]).name

meta_df = pd.DataFrame([c.metadata for c in all_chunks])
meta_df[["chunk_id", "source_name", "page", "content_type", "char_len"]].head(10)


,chunk_id,source_name,page,content_type,char_len
0,chunk_0000,Understanding_JEV_plain.pdf,0,text,659
1,chunk_0001,Understanding_JEV_plain.pdf,0,text,661
2,chunk_0002,Understanding_JEV_plain.pdf,0,text,699
3,chunk_0003,Understanding_JEV_plain.pdf,0,text,636
4,chunk_0004,Understanding_JEV_plain.pdf,0,text,412
5,chunk_0005,Understanding_JEV_plain.pdf,1,text,630
6,chunk_0006,Understanding_JEV_plain.pdf,1,text,635
7,chunk_0007,Understanding_JEV_plain.pdf,1,text,686
8,chunk_0008,Understanding_JEV_plain.pdf,1,text,629
9,chunk_0009,Understanding_JEV_plain.pdf,1,text,620


---
## 8. Document Indexing with Chroma

Chunks are embedded with the local `bge-small-en-v1.5` model and written into a persistent
Chroma collection. Persistence means the index survives a kernel restart -- re-running this cell
rebuilds from scratch, which is what we want for a reproducible notebook.

In [15]:
from langchain_chroma import Chroma
import shutil

PERSIST_DIR = "./chroma_db"
COLLECTION_NAME = "jev_rag_poc"

if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
)

print(f"Indexed {len(all_chunks)} chunks into Chroma collection '{COLLECTION_NAME}' at {PERSIST_DIR}")


Indexed 52 chunks into Chroma collection 'jev_rag_poc' at ./chroma_db


In [16]:
# Sanity check: does pure vector similarity retrieve something sensible?
probe = vectorstore.similarity_search("What does the Noul primitive return?", k=3)
for d in probe:
    print(f"[{d.metadata['chunk_id']}] ({d.metadata['content_type']}, p.{d.metadata['page']}) {d.page_content[:120]}...")


[chunk_0042] (text, p.9) without belonging to a fundamentally different class. A Score gives the
policy layer a quantity that can be thresholded ...
[chunk_0039] (text, p.8) judgments: Choice, Score, and Noul. Each primitive defines the shape
of the answer before inference begins, and the retu...
[chunk_0003] (text, p.0) JEV provides three core decision-making primitives that developers can assemble to introduce
explicit logic into retriev...


---
## 9. Hybrid Retrieval — BM25 (lexical) + Vector (semantic)

Vector similarity alone misses exact-term matches (model names, numeric benchmarks like "54%"
or "$0.00004"); BM25 alone misses paraphrases and synonyms. `EnsembleRetriever` runs both and
fuses their rankings, which is the standard "hybrid RAG" pattern -- this is the retrieval stage
*both* pipelines (with and without Jev) will share, so any difference measured later comes purely
from the reranking step.

In [17]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

HYBRID_TOP_K = 8      # candidates the hybrid retriever hands downstream
BM25_WEIGHT = 0.4
VECTOR_WEIGHT = 0.6

bm25_retriever = BM25Retriever.from_documents(all_chunks)
bm25_retriever.k = HYBRID_TOP_K

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": HYBRID_TOP_K})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[BM25_WEIGHT, VECTOR_WEIGHT],
)

print(f"Hybrid retriever ready: BM25(k={HYBRID_TOP_K}, w={BM25_WEIGHT}) + Vector(k={HYBRID_TOP_K}, w={VECTOR_WEIGHT})")


Hybrid retriever ready: BM25(k=8, w=0.4) + Vector(k=8, w=0.6)


In [ ]:
def hybrid_retrieve(query: str, k: int = HYBRID_TOP_K):
    t0 = time.perf_counter()
    docs = hybrid_retriever.invoke(query)[:k]
    latency = time.perf_counter() - t0
    return docs, latency

demo_query = "How does BM25 top-1 accuracy change when Jev is added as a reranker?"
demo_docs, demo_latency = hybrid_retrieve(demo_query)

print(f"Hybrid retrieval for: {demo_query!r}")
print(f"-> {len(demo_docs)} chunks in {demo_latency*1000:.1f} ms\n")
for d in demo_docs:
    print(f"[{d.metadata['chunk_id']}] ({d.metadata['content_type']}, p.{d.metadata['page']}) {d.page_content[:100]}...")


---
## 10. The Jev Reranker

As explained in the source material for this notebook: vector distance and BM25 score *lexical
and semantic proximity*, not *correctness*. A chunk can look relevant and still fail to answer
the question, or answer a different question that happens to share vocabulary.

**Jev's `Noul` primitive** evaluates a calibrated `P(criteria holds)` for each `(query, chunk)`
pair against an explicit true/false policy -- here, "does this chunk directly answer the
question" vs. "shares words/topic but doesn't answer it". Unlike a cross-encoder, the criteria
are plain instructions, not baked into a frozen model; unlike an LLM-as-reranker, cost and
latency stay flat per chunk instead of scaling with a full generative call.

In [ ]:
RERANK_CRITERIA = Noul(
    instructions="Does this passage directly and specifically answer the user's question?",
    criteria=NoulCriteria(
        true="states the specific fact, number, or mechanism the question asks for",
        false="only shares topic or vocabulary with the question, without answering it",
    ),
)

JEV_TOP_K = 3  # chunks kept after reranking, passed on to the LLM

print("Jev rerank policy (Noul criteria) defined.")


In [ ]:
def jev_rerank(query: str, docs: list[Document], top_k: int = JEV_TOP_K):
    t0 = time.perf_counter()
    states = [{"question": query, "candidate_chunk": d.page_content} for d in docs]
    responses = jev_fanout(states, {"relevance": RERANK_CRITERIA})
    scored = list(zip(docs, [r.answers["relevance"].noul for r in responses]))
    scored.sort(key=lambda pair: -pair[1])
    latency = time.perf_counter() - t0
    jev_tokens = sum(r.usage.input_tokens for r in responses)
    reranked_docs = [d for d, _ in scored[:top_k]]
    scores = [s for _, s in scored[:top_k]]
    return reranked_docs, scores, latency, jev_tokens

reranked_docs, reranked_scores, rerank_latency, rerank_tokens = jev_rerank(demo_query, demo_docs)

print(f"Jev reranked {len(demo_docs)} -> top {len(reranked_docs)} in {rerank_latency*1000:.1f} ms "
      f"({rerank_tokens} input tokens)\n")
for d, s in zip(reranked_docs, reranked_scores):
    print(f"  {s:.2f}  [{d.metadata['chunk_id']}] {d.page_content[:90]}...")


---
### 10b. Measuring Jev's Concurrency & Latency Scaling

Jev's core performance claim (from the source material) is that reranking cost and latency stay
close to flat as the candidate batch grows, because each `Noul` evaluation is a small, fixed-shape
call rather than a long generative one -- unlike an LLM-as-reranker, whose latency and cost scale
with prompt/output token volume. This is measured directly here, the same way the reference
benchmark notebook measures it: rerank the *same* candidate set at increasing worker-pool sizes
and watch wall-clock time.

In [ ]:
CONCURRENCY_LEVELS = [1, 2, 4, 8]

scaling_states = [{"question": demo_query, "candidate_chunk": d.page_content} for d in demo_docs]
scaling_results_ms = {}
for workers in CONCURRENCY_LEVELS:
    t0 = time.perf_counter()
    jev_fanout(scaling_states, {"relevance": RERANK_CRITERIA}, workers=workers)
    scaling_results_ms[workers] = (time.perf_counter() - t0) * 1000

for w, ms in scaling_results_ms.items():
    print(f"  {w:>2} worker(s): {ms:6.0f} ms for {len(scaling_states)} chunks")


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))

ws = list(scaling_results_ms.keys())
ms = list(scaling_results_ms.values())
a1.plot(ws, ms, marker="o", color=SERIES[0], lw=2.2, markersize=7, zorder=3)
a1.set_xticks(ws)
a1.set_xlabel("Worker pool size")
a1.set_ylabel("Wall-clock latency (ms)")
finish(a1, "Jev Rerank Latency vs. Concurrency", f"Reranking {len(scaling_states)} candidate chunks per run")

serial_ms = scaling_results_ms[1]
max_workers = max(CONCURRENCY_LEVELS)
parallel_ms = scaling_results_ms[max_workers]
speedup = serial_ms / parallel_ms
a2.barh(["Serial (1 worker)", f"{max_workers} workers"], [serial_ms, parallel_ms],
        color=[PALETTE["no_jev"], PALETTE["with_jev"]], height=0.5, zorder=2)
for i, v in enumerate([serial_ms, parallel_ms]):
    a2.text(v, i, f" {v:.0f} ms", va="center", ha="left", fontsize=9.5, weight="bold")
a2.set_xlabel("Wall-clock latency (ms) for one rerank call")
for s in ("top", "right", "left"):
    a2.spines[s].set_visible(False)
finish(a2, "Concurrency Speedup", f"{speedup:.1f}x faster at {max_workers} workers vs. serial")

plt.tight_layout()
plt.show()


---
## 11. Response Synthesis

The final answer is generated by Claude Sonnet (via OpenRouter) from whatever chunks the upstream
stage hands it -- this function doesn't know or care whether those chunks were Jev-reranked; it
just synthesizes an answer with citations from what it's given, which is exactly what makes the
"with Jev" vs "without Jev" comparison later a clean, apples-to-apples swap of one upstream stage.

In [ ]:
RAG_PROMPT_TEMPLATE = """Answer the question using ONLY the numbered context passages below.
Cite the passage number(s) you used in square brackets, e.g. [1]. If the context does not contain
the answer, say so explicitly instead of guessing.

Context:
{context}

Question: {question}

Answer:"""

def format_context(docs: list[Document]) -> str:
    parts = []
    for i, d in enumerate(docs, start=1):
        loc = f"{d.metadata['source_name']}, p.{d.metadata['page'] + 1}"
        parts.append(f"[{i}] ({loc}) {d.page_content}")
    return "\n\n".join(parts)

def synthesize_answer(query: str, docs: list[Document]):
    context = format_context(docs)
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=query)
    t0 = time.perf_counter()
    response = llm.invoke(prompt)
    latency = time.perf_counter() - t0
    usage = response.usage_metadata or {}
    return response.content, latency, usage, context

demo_answer, demo_gen_latency, demo_usage, demo_context = synthesize_answer(demo_query, reranked_docs)
print(demo_answer)
print(f"\n[generation: {demo_gen_latency*1000:.0f} ms | "
      f"tokens in={demo_usage.get('input_tokens')} out={demo_usage.get('output_tokens')}]")


---
### 11b. Cost Estimation

Latency and faithfulness aren't the whole performance picture -- cost is what makes reranking a
real economic decision, exactly as the source material's "Economics at Scale" analysis argues.
Two rates apply per query: the synthesis LLM's per-token price (Claude Sonnet, list pricing) and
Jev's per-token rate for `Noul` evaluations. `estimate_cost` turns raw token counts from any
pipeline run into a single USD figure.

In [ ]:
# Approximate list pricing; OpenRouter may add a small markup on top of these -- verify current
# rates at https://openrouter.ai/models before treating these as billing-accurate.
LLM_INPUT_PRICE_PER_1M = 3.00    # USD, Claude Sonnet input tokens
LLM_OUTPUT_PRICE_PER_1M = 15.00  # USD, Claude Sonnet output tokens
# JEV_RATE_PER_1M_INPUT was already defined in Section 2 (Jev client setup): $0.042 / 1M input tokens.

def estimate_cost(input_tokens: int, output_tokens: int, jev_tokens: int = 0) -> float:
    return (
        input_tokens / 1e6 * LLM_INPUT_PRICE_PER_1M
        + output_tokens / 1e6 * LLM_OUTPUT_PRICE_PER_1M
        + jev_tokens / 1e6 * JEV_RATE_PER_1M_INPUT
    )

demo_cost = estimate_cost(demo_usage.get("input_tokens", 0), demo_usage.get("output_tokens", 0), rerank_tokens)
print(f"Estimated cost for the demo query above: ${demo_cost:.5f}")


---
## 12. End-to-End Pipeline Functions

Two pipelines share the same retrieval stage and the same synthesis function; the only
difference is whether Jev reranks the candidates before they reach the LLM:

- **`pipeline_without_jev`** — hybrid retrieval's full `HYBRID_TOP_K` candidates go straight to
  the LLM. This is the realistic baseline: a hybrid RAG system with no reranking stage.
- **`pipeline_with_jev`** — hybrid retrieval's candidates are filtered down to `JEV_TOP_K` by the
  Jev reranker before synthesis.

This isolates exactly what Jev changes: fewer, higher-precision chunks in the context window.

In [ ]:
def pipeline_without_jev(query: str) -> dict:
    docs, retrieval_latency = hybrid_retrieve(query)
    answer, gen_latency, usage, context = synthesize_answer(query, docs)
    input_tokens, output_tokens = usage.get("input_tokens", 0), usage.get("output_tokens", 0)
    return {
        "query": query,
        "pipeline": "hybrid_no_jev",
        "answer": answer,
        "context": context,
        "docs_used": len(docs),
        "retrieval_latency_s": retrieval_latency,
        "rerank_latency_s": 0.0,
        "generation_latency_s": gen_latency,
        "total_latency_s": retrieval_latency + gen_latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "jev_tokens": 0,
        "cost_usd": estimate_cost(input_tokens, output_tokens, jev_tokens=0),
    }

def pipeline_with_jev(query: str) -> dict:
    docs, retrieval_latency = hybrid_retrieve(query)
    reranked, scores, rerank_latency, rerank_tok = jev_rerank(query, docs)
    answer, gen_latency, usage, context = synthesize_answer(query, reranked)
    input_tokens, output_tokens = usage.get("input_tokens", 0), usage.get("output_tokens", 0)
    return {
        "query": query,
        "pipeline": "hybrid_with_jev",
        "answer": answer,
        "context": context,
        "docs_used": len(reranked),
        "retrieval_latency_s": retrieval_latency,
        "rerank_latency_s": rerank_latency,
        "generation_latency_s": gen_latency,
        "total_latency_s": retrieval_latency + rerank_latency + gen_latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "jev_tokens": rerank_tok,
        "cost_usd": estimate_cost(input_tokens, output_tokens, jev_tokens=rerank_tok),
    }

print("Both pipeline variants defined.")


---
## 13. Evaluation Dataset

An 8-question set, each grounded in a specific, checkable fact from the two source PDFs. Having
a `gold_fact` per question lets the faithfulness judge (next section) check the answer against
ground truth, not just against whatever context happened to be retrieved.

In [ ]:
EVAL_SET = [
    {
        "question": "What are the three core decision-making primitives that JEV provides?",
        "gold_fact": "Binary Decision Making (the Noul primitive), Choice Ranker, and Score Ladder.",
    },
    {
        "question": "What calibrated output does the Noul primitive return?",
        "gold_fact": "A calibrated probability between 0 and 1 indicating whether a true/false criterion holds.",
    },
    {
        "question": "When should you use a Choice Ranker instead of a Score Ladder?",
        "gold_fact": "Choice Ranker isolates a single best option among discrete alternatives; Score Ladder reorders an entire collection into a ranked list.",
    },
    {
        "question": "According to the benchmark, how does BM25 top-1 accuracy change when Jev is added as a reranker?",
        "gold_fact": "BM25 baseline top-1 accuracy is 21%; adding Jev as a reranker raises it to 54%.",
    },
    {
        "question": "How much does 64-worker concurrency reduce processing time versus running decisions sequentially at 17/sec?",
        "gold_fact": "It drops processing time from about 40 seconds to about 7.6 seconds.",
    },
    {
        "question": "Why can't a cross-encoder reranker be steered with policy instructions the way Jev can?",
        "gold_fact": "Cross-encoders score a fixed (query, doc) pair with no instruction slot; prepending policy text to the query causes keyword collision that can promote exactly the documents the policy meant to exclude.",
    },
    {
        "question": "What two additional checks does JEV protect in production RAG systems, beyond reranking?",
        "gold_fact": "A pre-retrieval guardrail (screening prompts for safety hazards and severity) and a post-generation citation check (verifying cited passages support the claims).",
    },
    {
        "question": "What per-chunk cost and latency does the notebook claim for the Jev-based steerable reranker?",
        "gold_fact": "About $0.00004 per chunk, with sub-second latency.",
    },
]

print(f"{len(EVAL_SET)} evaluation questions loaded.")


---
## 14. Faithfulness Judge

Faithfulness here means: *is every claim in the generated answer actually supported by the
context it was given* -- not whether the answer happens to be true in general. An unfaithful
answer can still be factually correct by luck (or by the model's parametric knowledge); a
faithful RAG system's answers should be traceable to their retrieved context. Claude Sonnet
(via OpenRouter, temperature 0) grades this directly, returning a 0.0-1.0 score with a one-line
rationale.

In [ ]:
FAITHFULNESS_PROMPT = """You are grading a RAG system's answer for FAITHFULNESS: does the
answer's content follow strictly from the given context, with no unsupported claims?

Context given to the system:
{context}

Answer to grade:
{answer}

Score faithfulness from 0.0 (answer contains claims not in the context, or contradicts it) to
1.0 (every claim in the answer is directly supported by the context). Respond with ONLY a JSON
object: {{"score": <float 0-1>, "rationale": "<one sentence>"}}"""

def faithfulness_score(answer: str, context: str) -> tuple[float, str]:
    prompt = FAITHFULNESS_PROMPT.format(context=context, answer=answer)
    response = judge_llm.invoke(prompt)
    raw = response.content.strip()
    raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
        return float(parsed["score"]), parsed.get("rationale", "")
    except Exception:
        match = re.search(r"([01](?:\.\d+)?)", raw)
        return (float(match.group(1)) if match else 0.0), raw[:200]

demo_score, demo_rationale = faithfulness_score(demo_answer, demo_context)
print(f"Faithfulness: {demo_score:.2f} -- {demo_rationale}")


---
## 15. Run the Comparison: Hybrid RAG With vs. Without Jev

Every question in `EVAL_SET` runs through both pipelines. Results (latency breakdown, token
counts, faithfulness score) are collected into a single tidy `pandas.DataFrame` for analysis.

In [ ]:
results = []

for item in EVAL_SET:
    q = item["question"]

    r_base = pipeline_without_jev(q)
    r_base["faithfulness"], r_base["faithfulness_rationale"] = faithfulness_score(r_base["answer"], r_base["context"])
    results.append(r_base)

    r_jev = pipeline_with_jev(q)
    r_jev["faithfulness"], r_jev["faithfulness_rationale"] = faithfulness_score(r_jev["answer"], r_jev["context"])
    results.append(r_jev)

    print(f"done: {q[:60]}...")

results_df = pd.DataFrame(results)
results_df["total_tokens"] = results_df["input_tokens"] + results_df["output_tokens"]
results_df[["query", "pipeline", "docs_used", "total_latency_s", "total_tokens", "faithfulness"]]


---
## 16. Comparative Analysis

Four questions, four views of the same `results_df`, styled consistently with the benchmark
charts in `JEV_RERANKER_RAG.ipynb`:
1. Where does the time go, with vs. without Jev?
2. Does reranking actually cut tokens sent to the synthesis LLM?
3. Does the smaller, reranked context help or hurt faithfulness?
4. What does that translate to in dollars per query?

In [ ]:
summary = results_df.groupby("pipeline").agg(
    mean_retrieval_ms=("retrieval_latency_s", lambda s: s.mean() * 1000),
    mean_rerank_ms=("rerank_latency_s", lambda s: s.mean() * 1000),
    mean_generation_ms=("generation_latency_s", lambda s: s.mean() * 1000),
    mean_total_ms=("total_latency_s", lambda s: s.mean() * 1000),
    mean_input_tokens=("input_tokens", "mean"),
    mean_output_tokens=("output_tokens", "mean"),
    mean_total_tokens=("total_tokens", "mean"),
    mean_faithfulness=("faithfulness", "mean"),
    mean_docs_used=("docs_used", "mean"),
    mean_cost_usd=("cost_usd", "mean"),
).round(5)

pipelines = list(summary.index)
LABELS = {"hybrid_no_jev": "Hybrid (no Jev)", "hybrid_with_jev": "Hybrid + Jev rerank"}
COLORS = {"hybrid_no_jev": PALETTE["no_jev"], "hybrid_with_jev": PALETTE["with_jev"]}

summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
stages = ["mean_retrieval_ms", "mean_rerank_ms", "mean_generation_ms"]
stage_labels = ["Retrieval", "Rerank", "Generation"]
x = np.arange(len(stages))
width = 0.35

for i, pipeline in enumerate(pipelines):
    vals = summary.loc[pipeline, stages].values
    bars = ax.bar(x + (i - 0.5) * width, vals, width, label=LABELS[pipeline], color=COLORS[pipeline], zorder=2)
    ax.bar_label(bars, fmt="%.0f", padding=2, fontsize=8.5, weight="bold", color=PALETTE["ink"])

ax.set_xticks(x)
ax.set_xticklabels(stage_labels)
ax.set_ylabel("Mean latency (ms)")
ax.legend()
finish(ax, "Latency by Pipeline Stage", "Hybrid RAG with vs. without Jev reranking")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

totals_ms = [summary.loc[p, "mean_total_ms"] for p in pipelines]
draw_labeled_bars(axes[0], [LABELS[p] for p in pipelines], totals_ms,
                   colors=[COLORS[p] for p in pipelines], xlabel="Mean total latency (ms)", fmt="{:.0f} ms")
finish(axes[0], "Total Query Latency")

tokens = [summary.loc[p, "mean_total_tokens"] for p in pipelines]
reduction_pct = 100 * (1 - tokens[pipelines.index("hybrid_with_jev")] / tokens[pipelines.index("hybrid_no_jev")])
draw_labeled_bars(axes[1], [LABELS[p] for p in pipelines], tokens,
                   colors=[COLORS[p] for p in pipelines], xlabel="Mean tokens per query (context + completion)", fmt="{:.0f}")
finish(axes[1], "Token Usage", f"{reduction_pct:.0f}% reduction with Jev")

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
faith = [summary.loc[p, "mean_faithfulness"] for p in pipelines]
draw_labeled_bars(ax, [LABELS[p] for p in pipelines], faith,
                   colors=[COLORS[p] for p in pipelines], xlim=(0, 1.15),
                   threshold=0.7, threshold_label="Acceptable threshold (0.70)",
                   xlabel="Mean faithfulness score (LLM judge, 0-1)", fmt="{:.2f}")
finish(ax, "Response Faithfulness", "Hybrid RAG with vs. without Jev reranking")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4.5))

x = np.arange(len(pipelines))
costs = [summary.loc[p, "mean_cost_usd"] for p in pipelines]
bars = ax1.bar(x - 0.15, costs, width=0.3, color=[COLORS[p] for p in pipelines], zorder=3, label="Cost per query ($)")
for i, v in enumerate(costs):
    ax1.text(i - 0.15, v, f" ${v:.5f}", rotation=90, va="bottom", ha="center", fontsize=8.5, weight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels([LABELS[p] for p in pipelines])
ax1.set_ylabel("Mean cost per query (USD)")

ax2 = ax1.twinx()
lat = [summary.loc[p, "mean_total_ms"] for p in pipelines]
ax2.bar(x + 0.15, lat, width=0.3, color=PALETTE["muted"], alpha=0.45, zorder=2, label="Latency (ms)")
ax2.set_ylabel("Mean total latency (ms)", color=PALETTE["muted"])
ax2.grid(False)

finish(ax1, "Cost & Latency Economics per Query",
       "LLM synthesis dominates cost; Jev's own rerank cost is a rounding error by comparison")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for pipeline in pipelines:
    sub = results_df[results_df["pipeline"] == pipeline]
    ax.scatter(sub["total_latency_s"] * 1000, sub["faithfulness"],
               label=LABELS[pipeline], color=COLORS[pipeline], s=90, alpha=0.85, zorder=2)

ax.set_xlabel("Total latency (ms)")
ax.set_ylabel("Faithfulness score")
ax.legend()
finish(ax, "Latency vs. Faithfulness", "Per query, both pipelines")
plt.tight_layout()
plt.show()


In [ ]:
JEV_TOTAL_COST_USD = JEV_USAGE["input_tokens"] * JEV_RATE_PER_1M_INPUT / 1e6

print("--- Jev usage across this notebook run ---")
print(f"  Noul calls        : {JEV_USAGE['calls']}")
print(f"  Input tokens       : {JEV_USAGE['input_tokens']:,}")
print(f"  Mean call latency  : {JEV_USAGE['latency_s'] / max(JEV_USAGE['calls'], 1) * 1000:.0f} ms")
print(f"  Estimated Jev cost : ${JEV_TOTAL_COST_USD:.5f}")
print(f"\nJev cost as a fraction of one LLM synthesis call's token cost is negligible at this scale --")
print("the real payoff is upstream, in what it lets you NOT send to the (much pricier) synthesis LLM.")


---
## 17. Conclusions

- **Token usage**: Jev reranking cuts the number of chunks reaching the synthesis LLM from
  `HYBRID_TOP_K` down to `JEV_TOP_K`, directly reducing prompt tokens paid for on every query --
  see the reduction percentage printed above (Section 16).
- **Latency**: Jev reranking adds a small, roughly flat per-call overhead (the reranking step),
  but tends to *shrink* generation latency by shrinking the context the synthesis LLM has to read
  and reason over. The net effect on total latency depends on how large `HYBRID_TOP_K` is relative
  to `JEV_TOP_K`. Section 10b shows that overhead itself stays close to flat as worker concurrency
  increases -- the reranking stage scales, it doesn't become the bottleneck.
- **Cost**: the Section 16 economics chart makes the asymmetry explicit -- Jev's own per-call cost
  is negligible next to the synthesis LLM's token price; the real economic lever Jev pulls is
  upstream, by shrinking what gets sent to the (much pricier) generation call.
- **Faithfulness**: because Jev's `Noul` criterion explicitly screens for "does this chunk
  actually answer the question" rather than "is this chunk topically similar," the reranked
  context tends to contain less distracting, tangentially-related material -- which is the
  mechanism by which reranking can improve faithfulness, not just relevance ranking.
- **Where Jev doesn't help**: reranking is strictly a reordering/filtering step on top of
  whatever hybrid retrieval already found. If the gold-supporting chunk never makes it into the
  `HYBRID_TOP_K` candidate set, no reranker -- Jev or otherwise -- can produce a faithful answer.
  Retrieval recall remains the ceiling on end-to-end quality.

**To adapt this notebook to a larger corpus:** swap in more PDFs in Sections 3-4 (the image
captioning path already generalizes to any number of embedded images), raise `HYBRID_TOP_K` and
`JEV_TOP_K` proportionally, and expand `EVAL_SET` with questions grounded in the new material.


Real results from this run (8-question eval set)

```
┌────────────────────┬────────────────┬────────────────────────┐
│       Metric       │ Hybrid, no Jev │  Hybrid + Jev rerank   │
├────────────────────┼────────────────┼────────────────────────┤
│ Docs sent to LLM   │ 8              │ 3                      │
├────────────────────┼────────────────┼────────────────────────┤
│ Mean total tokens  │ 1,557          │ 684 (56% reduction)    │
├────────────────────┼────────────────┼────────────────────────┤
│ Mean cost/query    │ $0.00624       │ $0.00345 (45% cheaper) │
├────────────────────┼────────────────┼────────────────────────┤
│ Mean faithfulness  │ 0.875          │ 1.000                  │
├────────────────────┼────────────────┼────────────────────────┤
│ Mean total latency │ 3,612 ms       │ 4,001 ms               │
└────────────────────┴────────────────┴────────────────────────┘
```

Honest tradeoff worth flagging: Jev added net latency here (rerank cost ~570ms exceeded the ~180ms it saved on generation) even though it cut tokens/cost substantially and improved faithfulness to perfect — at HYBRID_TOP_K=8, Claude Sonnet's generation time doesn't shrink enough with fewer chunks to offset the rerank call itself. That's a real, not cherry-picked, finding from the run.